In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize

DATA_DIR = "/kaggle/input/datasets/mdfaysalhossan/jackfruit-leaf-dataset-v3/Jackfruit Leaf Dataset"
image_size = (224, 224)
batch_size = 32
epochs = 16
dropout_rate = 0.4
shuffle_buffer = 1024
elastic_alpha = 0.5
elastic_lambda = 1e-4

def compute_elastic_penalty(model: nn.Module, alpha: float, lam: float, exclude_bias: bool = True) -> torch.Tensor:
    l1 = torch.tensor(0., device=next(model.parameters()).device)
    l2 = torch.tensor(0., device=next(model.parameters()).device)
    for name, p in model.named_parameters():
        if not p.requires_grad or (exclude_bias and "bias" in name):
            continue
        l1 = l1 + p.abs().sum()
        l2 = l2 + p.pow(2).sum()
    return lam * (alpha * l1 + (1.0 - alpha) * 0.5 * l2)

if not os.path.exists(DATA_DIR):
    raise ValueError(f"Dataset path {DATA_DIR} does not exist! Check your Kaggle dataset path.")

print(f"Part 1 Complete — Elastic Net parameters: alpha={elastic_alpha}, lambda={elastic_lambda}")

In [ ]:
splits = ["Train", "Validation", "Test"]
split_paths, split_labels = {}, {}
class_names = sorted([d for d in os.listdir(os.path.join(DATA_DIR, "Train"))
                      if os.path.isdir(os.path.join(DATA_DIR, "Train", d))])
num_classes = len(class_names)
print(f"Classes: {class_names}, Number of classes: {num_classes}")

for split in splits:
    split_dir = os.path.join(DATA_DIR, split)
    if not os.path.isdir(split_dir):
        raise ValueError(f"{split} folder not found: {split_dir}")
    paths, labels = [], []
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(split_dir, class_name)
        if not os.path.isdir(class_dir):
            print(f"Warning: {class_name} not found in {split}")
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                paths.append(os.path.join(class_dir, fname))
                labels.append(class_idx)
    if not paths:
        raise ValueError(f"No images found in {split} folder!")
    split_paths[split], split_labels[split] = paths, labels

train_paths, train_labels = split_paths["Train"], split_labels["Train"]
val_paths, val_labels = split_paths["Validation"], split_labels["Validation"]
test_paths, test_labels = split_paths["Test"], split_labels["Test"]

print(f"Train: {len(train_paths)}, Validation: {len(val_paths)}, Test: {len(test_paths)}")
print("Part 2 Complete")

In [ ]:
from PIL import Image
import cv2, numpy as np
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

class JackfruitDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels, self.transform = paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as img:
            image = img.convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

class CLAHEGamma:
    def __init__(self, gamma=1.2, clipLimit=2.0, tileGridSize=(8, 8)):
        self.clipLimit, self.tileGridSize = clipLimit, tileGridSize
        invGamma = 1.0 / gamma
        self.table = np.array([
            ((i / 255.0) ** invGamma) * 255 for i in range(256)
        ]).astype("uint8")

    def __call__(self, img):
        lab = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(
            clipLimit=self.clipLimit,
            tileGridSize=self.tileGridSize
        ).apply(l)
        rgb = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
        return Image.fromarray(cv2.LUT(rgb, self.table))

transform = transforms.Compose([
    transforms.Resize(image_size),
    CLAHEGamma(gamma=1.2, clipLimit=2.0, tileGridSize=(8, 8)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

train_ds = JackfruitDataset(train_paths, train_labels, transform)
val_ds = JackfruitDataset(val_paths, val_labels, transform)
test_ds = JackfruitDataset(test_paths, test_labels, transform)

num_workers = 2
loader_args = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": num_workers > 0
}

train_loader = DataLoader(train_ds, shuffle=True, **loader_args)
val_loader = DataLoader(val_ds, shuffle=False, **loader_args)
test_loader = DataLoader(test_ds, shuffle=False, **loader_args)

for images, labels in train_loader:
    print(f"Images: {images.shape}, Labels: {labels.shape}")
    break

print("Part 3 Complete")

In [ ]:
import timm
import torch
import torch.nn as nn
from transformers import AutoModel

swin_model_name = "microsoft/swin-base-patch4-window7-224"

class DenseNet201_Swin_Hybrid(nn.Module):
    def __init__(self, num_classes):
        super(DenseNet201_Swin_Hybrid, self).__init__()

        self.cnn = timm.create_model(
            'densenet201',
            pretrained=True,
            num_classes=0
        )

        self.cnn_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.BatchNorm1d(1920),
            nn.Dropout(0.3),
            nn.Linear(1920, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3)
        )

        self.swin = AutoModel.from_pretrained(swin_model_name)

        swin_hidden = getattr(self.swin.config, "hidden_size", 1024)

        self.fusion_dim = 512 + swin_hidden
        self.fusion_bn = nn.BatchNorm1d(self.fusion_dim)
        self.fusion_dropout = nn.Dropout(0.3)
        self.fusion_linear = nn.Linear(self.fusion_dim, 512)
        self.fusion_relu = nn.ReLU()
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        feat_map = self.cnn.forward_features(x)
        cnn_vec = self.cnn_head(feat_map)

        swin_out = self.swin(pixel_values=x)

        if hasattr(swin_out, "pooler_output") and swin_out.pooler_output is not None:
            swin_vec = swin_out.pooler_output
        else:
            swin_vec = swin_out.last_hidden_state.mean(dim=1)

        fused = torch.cat((cnn_vec, swin_vec), dim=1)
        fused = self.fusion_bn(fused)
        fused = self.fusion_dropout(fused)
        fused = self.fusion_linear(fused)
        fused = self.fusion_relu(fused)

        logits = self.classifier(fused)
        return logits


model = DenseNet201_Swin_Hybrid(num_classes=num_classes)

for param in model.parameters():
    param.requires_grad = False

fine_tune_layers = 10
ct = 0
children = list(model.children())

for child in children:
    ct += 1
    if ct > len(children) - fine_tune_layers:
        for param in child.parameters():
            param.requires_grad = True

print("DenseNet201 + Swin Hybrid ready")
print(model)

In [ ]:
import torch, random, copy, numpy as np
import torch.nn as nn
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4, weight_decay=1e-5)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=3)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
amp_enabled = device.type == "cuda"
history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
best_val_loss = float('inf')
best_weights = None

for epoch in range(epochs):
    model.train()
    running_loss, correct = 0.0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} Train"):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(images)
            if logits.ndim != 2 or logits.shape[1] != num_classes:
                raise ValueError(f"Unexpected output shape: {logits.shape}")
            loss = loss_fn(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = correct / len(train_loader.dataset)
    model.eval()
    val_loss, val_correct = 0.0, 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} Val"):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                logits = model(images)
                loss = loss_fn(logits, labels)
            val_loss += loss.item() * images.size(0)
            val_correct += (logits.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader.dataset)
    val_acc = val_correct / len(val_loader.dataset)
    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    print(f"\nEpoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())
        print("Best model updated")

    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr != old_lr:
        print(f"LR reduced: {old_lr:.6g} -> {new_lr:.6g}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if best_weights is not None:
    model.load_state_dict(best_weights)
torch.save(model.state_dict(), "best_model.pth")
print("Training finished — best model saved")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix, classification_report

print("=== Part 6: Evaluating on Test Set ===")
model.eval()
test_loss, test_correct = 0.0, 0
preds, true_list = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        test_loss += loss.item() * images.size(0)
        test_correct += (outputs.argmax(1) == labels).sum().item()
        preds.append(outputs.cpu().numpy())
        true_list.append(labels.cpu().numpy())

test_loss /= len(test_loader.dataset)
test_acc = test_correct / len(test_loader.dataset)
y_pred = np.concatenate(preds)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.concatenate(true_list)

print(f"Final test loss: {test_loss:.4f}")
print(f"Final test accuracy: {test_acc:.4f}")

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_true, y_pred_classes, labels=range(num_classes))

FIG_W, FIG_H = 8, 7
TITLE_SIZE = 17
AXIS_LABEL_SIZE = 16
TICK_SIZE = 14
ANNOT_SIZE = 19

plt.figure(figsize=(FIG_W, FIG_H))
ax = sns.heatmap(
    cm, cmap='Greens', xticklabels=class_names, yticklabels=class_names,
    square=True, cbar=True, linewidths=0.75, linecolor='black', annot=False
)

threshold = cm.max() * 0.6
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = "white" if cm[i, j] > threshold else "black"
        ax.text(j + 0.5, i + 0.5, f"{cm[i, j]}", ha="center",
                va="center", fontsize=ANNOT_SIZE, color=color)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.3)
    spine.set_color('black')

cbar = ax.collections[0].colorbar
cbar.outline.set_visible(True)
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(1.3)

plt.title("Confusion Matrix", fontsize=TITLE_SIZE, pad=14)
plt.xlabel("Predicted values", fontsize=AXIS_LABEL_SIZE, labelpad=11)
plt.ylabel("True values", fontsize=AXIS_LABEL_SIZE, labelpad=11)
plt.xticks(fontsize=TICK_SIZE, rotation=45, ha='right')
plt.yticks(fontsize=TICK_SIZE, rotation=0)
plt.tight_layout()
plt.show()

print("\n=== Classification Report ===")
print(classification_report(
    y_true, y_pred_classes,
    labels=range(num_classes),
    target_names=class_names,
    zero_division=0,
    digits=4
))

print("=== Part 6 Complete ===")

In [ ]:
print("\n=== Part 7A: Training and Validation Loss ===")
plt.figure(figsize=(7, 5))
plt.plot(history['loss'], label='Train Loss', color='blue', linestyle='-', linewidth=2, marker='o', markersize=4)
plt.plot(history['val_loss'], label='Val Loss', color='red', linestyle='-', linewidth=2, marker='o', markersize=4)
plt.title('Training and Validation Loss', fontsize=16)
plt.xlabel('Epoch', fontsize=14)
plt.ylabel('Loss', fontsize=14)
plt.legend(frameon=True, fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(fontsize=13)
plt.yticks(fontsize=13)
plt.tight_layout()
plt.show()
print("Part 7A Complete")

print("\n=== Part 7B: Training and Validation Accuracy ===")
plt.figure(figsize=(7, 5))
plt.plot(history['accuracy'], label='Train Accuracy', color='blue', linestyle='-', linewidth=2, marker='o', markersize=4)
plt.plot(history['val_accuracy'], label='Val Accuracy', color='red', linestyle='-', linewidth=2, marker='o', markersize=4)
plt.title('Training and Validation Accuracy', fontsize=16)
plt.xlabel('Epoch', fontsize=14)
plt.ylabel('Accuracy', fontsize=14)
plt.legend(frameon=True, fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(fontsize=13)
plt.yticks(fontsize=13)
plt.tight_layout()
plt.show()
print("Part 7B Complete")

In [ ]:
import pandas as pd
import time
import torch
import torch.nn as nn

print("\n=== Part 9: Computational Cost Analysis ===")

def count_flops(model, input_tensor):
    flops_dict, hooks = {}, []
    def conv_hook(module, inp, out):
        b, ic = inp[0].shape[:2]
        oc, oh, ow = out.shape[1:]
        kh, kw = module.kernel_size
        flops_dict[id(module)] = b * (ic // module.groups) * oc * kh * kw * oh * ow * 2
        if module.bias is not None:
            flops_dict[id(module)] += b * oc * oh * ow
    def linear_hook(module, inp, out):
        b = inp[0].shape[0]
        flops_dict[id(module)] = b * module.in_features * module.out_features * 2
        if module.bias is not None:
            flops_dict[id(module)] += b * module.out_features
    def bn_hook(module, inp, out):
        flops_dict[id(module)] = inp[0].numel() * 2
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))
        elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            hooks.append(m.register_forward_hook(bn_hook))
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(input_tensor)
    for h in hooks:
        h.remove()
    model.train(was_training)
    return sum(flops_dict.values())

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_params = total_params - trainable_params

dummy_input = torch.randn(1, 3, image_size[0], image_size[1], device=device)
flops = count_flops(model, dummy_input)
gflops = flops / 1e9

model.eval()
repetitions, warmup = 100, 10
dummy_batch = torch.randn(batch_size, 3, image_size[0], image_size[1], device=device)

with torch.no_grad():
    for _ in range(warmup):
        model(dummy_batch)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repetitions):
        model(dummy_batch)
    if device.type == "cuda":
        torch.cuda.synchronize()
    inference_time = (time.perf_counter() - start) * 1000 / (repetitions * batch_size)

param_size = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
model_size = (param_size + buffer_size) / (1024 ** 2)

model_name = model.__class__.__name__
results = pd.DataFrame({
    "Model": [model_name],
    "Total Params": [f"{total_params:,}"],
    "Trainable": [f"{trainable_params:,}"],
    "Non-Trainable": [f"{non_trainable_params:,}"],
    "FLOPs": [f"{int(flops):,}"],
    "GFLOPs": [f"{gflops:.3f}"],
    "Inference Time (ms/image)": [f"{inference_time:.3f}"],
    "Model Size (MB)": [f"{model_size:.2f}"]
})

display(results)
print("Part 9 Complete")

In [ ]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import t
from tqdm import tqdm
import gc
import copy

NUM_FOLDS = 5
BATCH_SIZE_CV = batch_size if "batch_size" in globals() else 32
EPOCHS_CV = epochs if "epochs" in globals() else 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cv_paths = list(train_paths)
cv_labels = np.asarray(train_labels, dtype=np.int64)
assert len(cv_paths) == len(cv_labels), "Number of image paths and labels do not match."

print("Images available for 5-fold CV:", len(cv_paths))
print("Number of classes:", num_classes)
print("Device:", device)

skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(cv_labels)), cv_labels), start=1):
    print(f"\n{'=' * 55}")
    print(f"Fold {fold}/{NUM_FOLDS}")
    print(f"Training images: {len(train_idx)}")
    print(f"Validation images: {len(val_idx)}")
    print(f"{'=' * 55}")

    fold_train_ds = JackfruitDataset(
        [cv_paths[i] for i in train_idx],
        cv_labels[train_idx].tolist(),
        transform=transform
    )
    fold_val_ds = JackfruitDataset(
        [cv_paths[i] for i in val_idx],
        cv_labels[val_idx].tolist(),
        transform=transform
    )
    fold_train_loader = DataLoader(
        fold_train_ds, batch_size=BATCH_SIZE_CV, shuffle=True,
        num_workers=2, pin_memory=torch.cuda.is_available()
    )
    fold_val_loader = DataLoader(
        fold_val_ds, batch_size=BATCH_SIZE_CV, shuffle=False,
        num_workers=2, pin_memory=torch.cuda.is_available()
    )

    model_cv = DenseNet201_Swin_Hybrid(num_classes=num_classes).to(device)
    for param in model_cv.parameters():
        param.requires_grad = False
    for module in [
        model_cv.cnn_head,
        model_cv.fusion_bn,
        model_cv.fusion_dropout,
        model_cv.fusion_linear,
        model_cv.fusion_relu,
        model_cv.classifier
    ]:
        for param in module.parameters():
            param.requires_grad = True

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model_cv.parameters()),
        lr=3e-4, weight_decay=1e-5
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.3, patience=3
    )
    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model_cv.state_dict())
    patience = 5
    no_improve = 0

    for epoch in range(EPOCHS_CV):
        model_cv.train()
        total_loss = 0.0
        for images, targets in tqdm(
            fold_train_loader,
            desc=f"Fold {fold} | Epoch {epoch+1}/{EPOCHS_CV}",
            leave=False
        ):
            images = images.to(device)
            targets = targets.to(device).long()
            optimizer.zero_grad(set_to_none=True)
            logits = model_cv(images)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)

        train_loss = total_loss / len(fold_train_ds)
        model_cv.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for images, targets in fold_val_loader:
                images = images.to(device)
                targets = targets.to(device).long()
                logits = model_cv(images)
                loss = criterion(logits, targets)
                val_loss_total += loss.item() * images.size(0)

        val_loss = val_loss_total / len(fold_val_ds)
        scheduler.step(val_loss)
        print(
            f"Fold {fold} | Epoch {epoch+1} "
            f"| Train Loss: {train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f}"
        )
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model_cv.state_dict())
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            print("Early stopping triggered.")
            break

    model_cv.load_state_dict(best_weights)
    model_cv.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, targets in fold_val_loader:
            images = images.to(device)
            logits = model_cv(images)
            predictions = logits.argmax(dim=1).cpu().numpy()
            y_pred.extend(predictions.tolist())
            y_true.extend(targets.numpy().tolist())

    fold_accuracy = accuracy_score(y_true, y_pred) * 100
    fold_macro_f1 = f1_score(
        y_true, y_pred, average="macro", zero_division=0
    ) * 100
    fold_results.append({
        "Fold": fold,
        "Accuracy (%)": fold_accuracy,
        "Macro F1 (%)": fold_macro_f1
    })
    print(
        f"Fold {fold} Results | "
        f"Accuracy: {fold_accuracy:.2f}% | "
        f"Macro F1: {fold_macro_f1:.2f}%"
    )

    del model_cv, optimizer, scheduler
    del fold_train_loader, fold_val_loader
    del fold_train_ds, fold_val_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(fold_results)

def calculate_ci95(values):
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean_value = np.mean(values)
    std_value = np.std(values, ddof=1)
    t_critical = t.ppf(0.975, df=n - 1)
    margin = t_critical * std_value / np.sqrt(n)
    return {
        "Mean (%)": mean_value,
        "Std ± (%)": std_value,
        "95% CI Lower (%)": mean_value - margin,
        "95% CI Upper (%)": mean_value + margin
    }

summary_df = pd.DataFrame({
    "Accuracy": calculate_ci95(results_df["Accuracy (%)"]),
    "Macro F1": calculate_ci95(results_df["Macro F1 (%)"])
}).T

print("\n========== Fold-wise Results ==========")
print(results_df.round(2).to_string(index=False))
print("\n========== Overall Performance with 95% CI ==========")
print(summary_df.round(2).to_string())

results_df.to_csv("Part10_5Fold_Results.csv", index=False)
summary_df.to_csv("Part10_95CI_Summary.csv")
print("\nPart 10 completed successfully.")
print("Saved: Part10_5Fold_Results.csv")
print("Saved: Part10_95CI_Summary.csv")


In [ ]:

import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage.segmentation import slic

model.eval()

if 'class_names' not in globals():
    raise RuntimeError("Run the dataset/class mapping cell first.")

num_classes = len(class_names)
print("Classes:", class_names)


class DenseNetSwinGradCAMPlusPlus:
    def __init__(self, model):
        self.model = model
        self.cnn_activation = None
        self.swin_activation = None

        self.cnn_handle = model.cnn.features.register_forward_hook(
            self.cnn_hook
        )
        self.swin_handle = model.swin.layernorm.register_forward_hook(
            self.swin_hook
        )

    def cnn_hook(self, module, inputs, output):
        self.cnn_activation = output

    def swin_hook(self, module, inputs, output):
        self.swin_activation = output

    def remove_hooks(self):
        self.cnn_handle.remove()
        self.swin_handle.remove()

    def prepare_feature_map(self, feature, branch):
        if feature.dim() == 4:
            if branch == "cnn":
                return feature

            hidden = getattr(self.model.swin.config, "hidden_size", None)

            # Hugging Face Swin may output [B,H,W,C]
            if hidden is not None and feature.shape[-1] == hidden:
                return feature.permute(0, 3, 1, 2).contiguous()

            if hidden is not None and feature.shape[1] == hidden:
                return feature

            raise ValueError(
                f"Unexpected {branch} feature shape: {tuple(feature.shape)}"
            )

        if feature.dim() == 3:
            b, tokens, channels = feature.shape
            side = int(np.sqrt(tokens))
            if side * side != tokens:
                raise ValueError(
                    f"Cannot reshape {tokens} tokens into a square map. "
                    f"Actual shape: {tuple(feature.shape)}"
                )
            return feature.transpose(1, 2).reshape(
                b, channels, side, side
            )

        raise ValueError(
            f"Unsupported {branch} feature shape: {tuple(feature.shape)}"
        )

    def gradcam_pp(self, activation, gradients, branch):
        activation = self.prepare_feature_map(activation, branch)
        gradients = self.prepare_feature_map(gradients, branch)

        # Grad-CAM++ alpha weights
        grad2 = gradients.pow(2)
        grad3 = gradients.pow(3)
        activation_sum = activation.sum(
            dim=(2, 3), keepdim=True
        )

        denominator = (
            2.0 * grad2
            + activation_sum * grad3
        )
        denominator = torch.where(
            denominator.abs() > 1e-8,
            denominator,
            torch.full_like(denominator, 1e-8)
        )

        alpha = grad2 / denominator
        positive_gradients = torch.relu(gradients)
        weights = (alpha * positive_gradients).sum(
            dim=(2, 3), keepdim=True
        )

        cam = torch.relu((weights * activation).sum(dim=1))
        return cam[0].detach().cpu().numpy()

    def generate(self, image_tensor, target_class):
        self.model.zero_grad(set_to_none=True)

        with torch.enable_grad():
            output = self.model(image_tensor)
            if isinstance(output, tuple):
                output = output[0]

            score = output[0, target_class]
            cnn_activation = self.cnn_activation
            swin_activation = self.swin_activation

            cnn_grad, swin_grad = torch.autograd.grad(
                score,
                [cnn_activation, swin_activation],
                retain_graph=False,
                create_graph=False
            )

            cnn_cam = self.gradcam_pp(
                cnn_activation, cnn_grad, "cnn"
            )
            swin_cam = self.gradcam_pp(
                swin_activation, swin_grad, "swin"
            )

        return cnn_cam, swin_cam


def normalize_cam(cam):
    cam = np.nan_to_num(
        cam, nan=0.0, posinf=0.0, neginf=0.0
    )
    cam = np.maximum(cam, 0)
    maximum = cam.max()
    return cam / maximum if maximum > 1e-8 else cam


def resize_cam(cam, shape):
    h, w = shape[:2]
    cam = cv2.resize(
        cam, (w, h), interpolation=cv2.INTER_CUBIC
    )
    return normalize_cam(cam)


def create_heatmap(cam):
    heatmap = (np.clip(cam, 0, 1) * 255).astype(np.uint8)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    return cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)


def create_visual(image, cam):
    heatmap = create_heatmap(cam)
    result = (
        0.25 * image.astype(np.float32)
        + 0.75 * heatmap.astype(np.float32)
    )
    return np.clip(result, 0, 255).astype(np.uint8)


def create_combined_visual(image, positive, negative):
    ph = create_heatmap(positive)
    nh = create_heatmap(negative)
    result = (
        0.20 * image.astype(np.float32)
        + 0.40 * ph.astype(np.float32)
        + 0.40 * nh.astype(np.float32)
    )
    return np.clip(result, 0, 255).astype(np.uint8)


def create_slic_segments(image, n_segments=50):
    return slic(
        image.astype(np.float32) / 255.0,
        n_segments=n_segments,
        compactness=12,
        sigma=1,
        start_label=1,
        channel_axis=2
    )


def calculate_segment_scores(segments, cam):
    results = []
    for sid in np.unique(segments):
        values = cam[segments == sid]
        if values.size:
            results.append((int(sid), float(values.mean())))
    return sorted(results, key=lambda x: x[1], reverse=True)[:8]

gradcam_pp = DenseNetSwinGradCAMPlusPlus(model)
selected_indices = []

try:
    for class_idx in range(num_classes):
        indices = [
            i for i in range(len(test_ds))
            if int(test_ds[i][1]) == class_idx
        ]
        if not indices:
            raise ValueError(
                f"No test image found for class {class_names[class_idx]}"
            )
        selected_indices.append(int(np.random.choice(indices)))

    fig = plt.figure(figsize=(19, 4.0 * num_classes))
    gs = fig.add_gridspec(
        num_classes, 5,
        width_ratios=[1.05, 1.05, 1.05, 1.05, 1.8],
        hspace=0.25, wspace=0.10
    )

    for row, idx in enumerate(selected_indices):
        image_tensor, true_label = test_ds[idx]
        true_label = int(true_label)
        input_tensor = image_tensor.unsqueeze(0).float().to(device)

        image = image_tensor.detach().cpu().permute(1, 2, 0).numpy()
        image = np.clip(image * 0.5 + 0.5, 0, 1)
        image_uint8 = np.ascontiguousarray(
            (image * 255).astype(np.uint8)
        )

        with torch.no_grad():
            output = model(input_tensor)
            if isinstance(output, tuple):
                output = output[0]
            probabilities = torch.softmax(output, dim=1)[0]
            pred_class = int(probabilities.argmax().item())
            confidence = float(probabilities[pred_class].item())

        cnn_cam, swin_cam = gradcam_pp.generate(
            input_tensor, pred_class
        )
        cnn_cam = resize_cam(cnn_cam, image_uint8.shape)
        swin_cam = resize_cam(swin_cam, image_uint8.shape)

        hybrid = normalize_cam((cnn_cam + swin_cam) / 2.0)
        positive_cam = hybrid.copy()

        negative_cam = normalize_cam(1.0 - hybrid)
        negative_cam[hybrid > 0.5] = 0

        segments = create_slic_segments(image_uint8, 50)
        top_segments = calculate_segment_scores(segments, hybrid)

        positive_visual = create_visual(image_uint8, positive_cam)
        negative_visual = create_visual(image_uint8, negative_cam)
        combined_visual = create_combined_visual(
            image_uint8, positive_cam, negative_cam
        )

        ax1, ax2, ax3, ax4, ax5 = [
            fig.add_subplot(gs[row, col]) for col in range(5)
        ]

        for ax, shown in zip(
            [ax1, ax2, ax3, ax4],
            [
                image_uint8, positive_visual,
                negative_visual, combined_visual
            ]
        ):
            ax.imshow(shown)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_aspect('equal', adjustable='box')

        ax1.set_title(
            class_names[true_label], fontsize=12,
            fontweight='bold', pad=6
        )
        ax2.set_title(
            "Grad-CAM++ — Positive", fontsize=11,
            fontweight='bold', color='green', pad=6
        )
        ax3.set_title(
            "Grad-CAM++ — Negative", fontsize=11,
            fontweight='bold', color='red', pad=6
        )
        ax4.set_title(
            "Grad-CAM++ Explanation", fontsize=11,
            fontweight='bold', pad=6
        )

        # Prediction details
        ax5.axis('off')
        ax5.set_title(
            "Prediction Details", fontsize=12,
            fontweight='bold', pad=6
        )

        correct = pred_class == true_label
        result = "Correct" if correct else "Incorrect"
        result_color = 'green' if correct else 'red'

        details = [
            ("True label:", class_names[true_label], 'black'),
            ("Predicted:", class_names[pred_class], result_color),
            ("Confidence:", f"{confidence * 100:.2f}%", 'black'),
            ("Result:", result, result_color)
        ]

        y = 0.95
        for label, value, color in details:
            ax5.text(
                0.02, y, label, fontsize=10,
                fontweight='bold', transform=ax5.transAxes, va='top'
            )
            ax5.text(
                0.43, y, value, fontsize=10,
                fontweight='bold', color=color,
                transform=ax5.transAxes, va='top'
            )
            y -= 0.105

        ax5.text(
            0.02, 0.49, "Top contributing regions:",
            fontsize=10, fontweight='bold',
            transform=ax5.transAxes, va='top'
        )

        y = 0.40
        for rank, (sid, score) in enumerate(top_segments, 1):
            ax5.text(
                0.04, y, f"{rank}. Segment {sid}",
                fontsize=9, transform=ax5.transAxes, va='top'
            )
            ax5.text(
                0.62, y, f"+{score:.4f}",
                fontsize=9, fontweight='bold', color='green',
                transform=ax5.transAxes, va='top'
            )
            y -= 0.047

    fig.subplots_adjust(
        left=0.015, right=0.99, top=0.97, bottom=0.02
    )
    plt.show()

finally:
    gradcam_pp.remove_hooks()

print("\n" + "=" * 65)
print("DENSENET201 + SWIN TRANSFORMER GRAD-CAM++ COMPLETE")
print("=" * 65)

for row, idx in enumerate(selected_indices):
    print(f"{class_names[row]} -> Test index: {idx}")


In [ ]:
import numpy as np
import pandas as pd
import torch
import cv2
from scipy import stats
from tqdm.auto import tqdm

model.eval()
gradcam_pp = DenseNetSwinGradCAMPlusPlus(model)

ACTIVATION_THRESHOLD = 0.5
results = []

try:
    for idx in tqdm(range(len(test_ds)), desc="Grad-CAM++ Statistics"):
        image_tensor, true_label = test_ds[idx]
        true_label = int(true_label)
        input_tensor = image_tensor.unsqueeze(0).float().to(device)

        with torch.no_grad():
            output = model(input_tensor)
            if isinstance(output, tuple):
                output = output[0]
            pred_class = int(output.argmax(dim=1).item())

        cnn_cam, swin_cam = gradcam_pp.generate(input_tensor, pred_class)

        h, w = image_tensor.shape[-2:]
        cnn_cam = cv2.resize(cnn_cam, (w, h), interpolation=cv2.INTER_CUBIC)
        swin_cam = cv2.resize(swin_cam, (w, h), interpolation=cv2.INTER_CUBIC)

        cnn_cam = normalize_cam(cnn_cam)
        swin_cam = normalize_cam(swin_cam)
        hybrid_cam = normalize_cam((cnn_cam + swin_cam) / 2.0)

        activation_area = np.mean(
            hybrid_cam >= ACTIVATION_THRESHOLD
        ) * 100
        mean_intensity = float(np.mean(hybrid_cam))
        maximum_activation = float(np.max(hybrid_cam))

        results.append({
            "Image_Index": idx,
            "True_Class": class_names[true_label],
            "Predicted_Class": class_names[pred_class],
            "Correct": int(pred_class == true_label),
            "Activation_Area (%)": activation_area,
            "Mean_Heatmap_Intensity": mean_intensity,
            "Maximum_Activation": maximum_activation
        })

finally:
    gradcam_pp.remove_hooks()

image_results_df = pd.DataFrame(results)

metrics = [
    ("Activation Area (%)", "Activation_Area (%)"),
    ("Mean Heatmap Intensity", "Mean_Heatmap_Intensity"),
    ("Maximum Activation", "Maximum_Activation")
]

summary = []

for metric_name, column in metrics:
    values = image_results_df[column].dropna().to_numpy()
    n = len(values)
    mean = np.mean(values)
    sd = np.std(values, ddof=1) if n > 1 else np.nan

    if n > 1:
        margin = stats.t.ppf(0.975, df=n - 1) * sd / np.sqrt(n)
        ci_lower = mean - margin
        ci_upper = mean + margin
    else:
        ci_lower, ci_upper = np.nan, np.nan

    summary.append({
        "Metric": metric_name,
        "N": n,
        "Mean": mean,
        "SD": sd,
        "95% CI Lower": ci_lower,
        "95% CI Upper": ci_upper
    })

table1_df = pd.DataFrame(summary)

print(table1_df.round(4).to_string(index=False))

image_results_df.to_csv(
    "GradCAM_PP_Image_Level_Results.csv", index=False
)
table1_df.to_csv(
    "GradCAM_PP_Overall_Statistics.csv", index=False
)

print("\nSaved: GradCAM_PP_Image_Level_Results.csv")
print("Saved: GradCAM_PP_Overall_Statistics.csv")


In [ ]:
import numpy as np
import pandas as pd
import torch
import cv2
from tqdm.auto import tqdm

model.eval()
gradcam_pp = DenseNetSwinGradCAMPlusPlus(model)
ACTIVATION_THRESHOLD = 0.5
results = []

try:
    for idx in tqdm(range(len(test_ds)), desc="Class-wise Grad-CAM++"):
        image_tensor, true_label = test_ds[idx]
        true_label = int(true_label)
        input_tensor = image_tensor.unsqueeze(0).float().to(device)

        with torch.no_grad():
            output = model(input_tensor)
            if isinstance(output, tuple):
                output = output[0]
            pred_class = int(output.argmax(dim=1).item())

        cnn_cam, swin_cam = gradcam_pp.generate(input_tensor, pred_class)
        h, w = image_tensor.shape[-2:]

        cnn_cam = cv2.resize(cnn_cam, (w, h), interpolation=cv2.INTER_CUBIC)
        swin_cam = cv2.resize(swin_cam, (w, h), interpolation=cv2.INTER_CUBIC)

        cnn_cam = normalize_cam(cnn_cam)
        swin_cam = normalize_cam(swin_cam)
        hybrid_cam = normalize_cam((cnn_cam + swin_cam) / 2.0)

        activation_area = np.mean(
            hybrid_cam >= ACTIVATION_THRESHOLD
        ) * 100
        mean_intensity = float(np.mean(hybrid_cam))

        results.append({
            "Image_Index": idx,
            "True_Class": class_names[true_label],
            "Predicted_Class": class_names[pred_class],
            "Correct": int(pred_class == true_label),
            "Activation_Area (%)": activation_area,
            "Mean_Heatmap_Intensity": mean_intensity
        })

finally:
    gradcam_pp.remove_hooks()

image_results_df = pd.DataFrame(results)

classwise = image_results_df.groupby("True_Class").agg(
    N=("Image_Index", "count"),
    Area_Mean=("Activation_Area (%)", "mean"),
    Area_SD=("Activation_Area (%)", "std"),
    Intensity_Mean=("Mean_Heatmap_Intensity", "mean"),
    Intensity_SD=("Mean_Heatmap_Intensity", "std")
)

table2_df = classwise.reindex(class_names).reset_index()
table2_df = table2_df.rename(columns={"True_Class": "Class"})

table2_df["Activation Area Mean ± SD (%)"] = table2_df.apply(
    lambda r: f"{r['Area_Mean']:.2f} ± {r['Area_SD']:.2f}"
    if pd.notna(r["Area_SD"]) else f"{r['Area_Mean']:.2f} ± NA",
    axis=1
)
table2_df["Heatmap Intensity Mean ± SD"] = table2_df.apply(
    lambda r: f"{r['Intensity_Mean']:.4f} ± {r['Intensity_SD']:.4f}"
    if pd.notna(r["Intensity_SD"])
    else f"{r['Intensity_Mean']:.4f} ± NA",
    axis=1
)

table2_df = table2_df[[
    "Class", "N",
    "Activation Area Mean ± SD (%)",
    "Heatmap Intensity Mean ± SD"
]]

print(table2_df.to_string(index=False))

table2_df.to_csv("GradCAM_PP_Classwise_Statistics.csv", index=False)
image_results_df.to_csv("GradCAM_PP_Classwise_Image_Level.csv", index=False)

print("\nSaved: GradCAM_PP_Classwise_Statistics.csv")
print("Saved: GradCAM_PP_Classwise_Image_Level.csv")


In [ ]:
import numpy as np
import pandas as pd
import torch
import cv2
from scipy import stats
from sklearn.metrics import auc
from skimage.segmentation import slic
from tqdm.auto import tqdm

model.eval()
gradcam_pp = DenseNetSwinGradCAMPlusPlus(model)

NUM_SEGMENTS = 50
NUM_STEPS = 10
BASELINE_VALUE = 0
image_results = []

def denormalize_image(tensor):
    img = tensor.detach().cpu().permute(1, 2, 0).numpy()
    img = np.clip(img * 0.5 + 0.5, 0, 1)
    return np.ascontiguousarray((img * 255).astype(np.uint8))

def image_to_tensor(image):
    img = torch.from_numpy(
        image.astype(np.float32) / 255.0
    ).permute(2, 0, 1)
    img = (img - 0.5) / 0.5
    return img.unsqueeze(0).to(device)

try:
    for idx in tqdm(range(len(test_ds)), desc="Grad-CAM++ Deletion/Insertion"):
        image_tensor, true_label = test_ds[idx]
        true_label = int(true_label)
        input_tensor = image_tensor.unsqueeze(0).float().to(device)
        image = denormalize_image(image_tensor)
        h, w = image.shape[:2]

        with torch.no_grad():
            output = model(input_tensor)
            if isinstance(output, tuple):
                output = output[0]
            probabilities = torch.softmax(output, dim=1)[0]
            pred_class = int(probabilities.argmax().item())
            original_conf = float(probabilities[pred_class].item())

        cnn_cam, swin_cam = gradcam_pp.generate(input_tensor, pred_class)
        cnn_cam = normalize_cam(
            cv2.resize(cnn_cam, (w, h), interpolation=cv2.INTER_CUBIC)
        )
        swin_cam = normalize_cam(
            cv2.resize(swin_cam, (w, h), interpolation=cv2.INTER_CUBIC)
        )
        cam = normalize_cam((cnn_cam + swin_cam) / 2.0)

        segments = slic(
            image.astype(np.float32) / 255.0,
            n_segments=NUM_SEGMENTS,
            compactness=12,
            sigma=1,
            start_label=1,
            channel_axis=2
        )
        ranked_segments = sorted(
            [(int(sid), float(cam[segments == sid].mean()))
             for sid in np.unique(segments)],
            key=lambda item: item[1],
            reverse=True
        )
        ranked_ids = [sid for sid, _ in ranked_segments]
        n_segments = len(ranked_ids)

        fractions = np.linspace(0, 1, NUM_STEPS + 1)
        deletion_scores = [original_conf]

        insertion_image = np.full_like(image, BASELINE_VALUE)
        deletion_image = image.copy()

        with torch.no_grad():
            base_tensor = image_to_tensor(insertion_image)
            base_output = model(base_tensor)
            if isinstance(base_output, tuple):
                base_output = base_output[0]
            insertion_scores = [
                float(torch.softmax(base_output, dim=1)[0, pred_class].item())
            ]

        for step in range(1, NUM_STEPS + 1):
            start = int((step - 1) * n_segments / NUM_STEPS)
            end = int(step * n_segments / NUM_STEPS)
            ids = ranked_ids[start:end]

            for sid in ids:
                mask = segments == sid
                deletion_image[mask] = BASELINE_VALUE
                insertion_image[mask] = image[mask]

            deletion_tensor = image_to_tensor(deletion_image)
            insertion_tensor = image_to_tensor(insertion_image)

            with torch.no_grad():
                out_del = model(deletion_tensor)
                out_ins = model(insertion_tensor)
                if isinstance(out_del, tuple):
                    out_del = out_del[0]
                if isinstance(out_ins, tuple):
                    out_ins = out_ins[0]

                deletion_scores.append(
                    float(torch.softmax(out_del, dim=1)[0, pred_class].item())
                )
                insertion_scores.append(
                    float(torch.softmax(out_ins, dim=1)[0, pred_class].item())
                )

        deletion_auc = auc(fractions, deletion_scores)
        insertion_auc = auc(fractions, insertion_scores)
        confidence_drop = (original_conf - deletion_scores[-1]) * 100

        image_results.append({
            "Image_Index": idx,
            "True_Class": class_names[true_label],
            "Predicted_Class": class_names[pred_class],
            "Correct": int(pred_class == true_label),
            "Original_Confidence (%)": original_conf * 100,
            "Deletion_AUC": deletion_auc,
            "Insertion_AUC": insertion_auc,
            "Confidence_Drop_After_Deletion (%)": confidence_drop
        })

finally:
    gradcam_pp.remove_hooks()

results_df = pd.DataFrame(image_results)
results_df.to_csv("GradCAM_PP_Deletion_Insertion_Image_Level.csv", index=False)

metrics = [
    ("Deletion AUC", "Deletion_AUC"),
    ("Insertion AUC", "Insertion_AUC"),
    ("Confidence Drop after Deletion (%)",
     "Confidence_Drop_After_Deletion (%)")
]

summary = []
for metric_name, column in metrics:
    values = results_df[column].dropna().to_numpy()
    n = len(values)
    mean = np.mean(values)
    sd = np.std(values, ddof=1) if n > 1 else np.nan

    if n > 1:
        margin = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
        lower, upper = mean - margin, mean + margin
    else:
        lower, upper = np.nan, np.nan

    summary.append({
        "Metric": metric_name,
        "N": n,
        "Mean": mean,
        "SD": sd,
        "95% CI Lower": lower,
        "95% CI Upper": upper
    })

table3_df = pd.DataFrame(summary)
print(table3_df.round(4).to_string(index=False))

table3_df.to_csv("GradCAM_PP_Deletion_Insertion_Summary.csv", index=False)
print("\nSaved: GradCAM_PP_Deletion_Insertion_Image_Level.csv")
print("Saved: GradCAM_PP_Deletion_Insertion_Summary.csv")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2

model.eval()

class_names = [
    'Blight', 'Dried', 'Fungal Spot', 'Healthy',
    'Leaf Miner', 'Pest Damage', 'Senescence', 'Young'
]
num_classes = len(class_names)

def predict(model, x):
    with torch.no_grad():
        output = model(x)
        if isinstance(output, tuple):
            output = output[0]
        return torch.softmax(output, dim=1)

def occlusion_sensitivity(
    model, image_tensor, patch_size=32, stride=16, batch_size=32
):
    x = image_tensor.unsqueeze(0).float().to(device)
    base_prob = predict(model, x)
    pred_class = int(base_prob[0].argmax().item())
    base_score = float(base_prob[0, pred_class].item())

    _, _, h, w = x.shape
    patch_size = min(patch_size, h, w)

    ys = list(range(0, h - patch_size + 1, stride))
    xs = list(range(0, w - patch_size + 1, stride))
    if not ys or ys[-1] != h - patch_size:
        ys.append(h - patch_size)
    if not xs or xs[-1] != w - patch_size:
        xs.append(w - patch_size)

    ys, xs = sorted(set(ys)), sorted(set(xs))
    positions = [(y, x0) for y in ys for x0 in xs]
    row_map = {v: i for i, v in enumerate(ys)}
    col_map = {v: i for i, v in enumerate(xs)}
    raw_heatmap = np.zeros((len(ys), len(xs)), dtype=np.float32)

    original = x[0]
    batches, batch_positions = [], []

    for y, x0 in positions:
        occluded = original.clone()
        region = occluded[:, y:y+patch_size, x0:x0+patch_size]
        region_mean = region.mean(dim=(1, 2), keepdim=True)
        occluded[:, y:y+patch_size, x0:x0+patch_size] = region_mean
        batches.append(occluded)
        batch_positions.append((y, x0))

    for start in range(0, len(batches), batch_size):
        batch = torch.stack(batches[start:start+batch_size]).to(device)

        with torch.no_grad():
            output = model(batch)
            if isinstance(output, tuple):
                output = output[0]
            scores = torch.softmax(output, dim=1)[:, pred_class]

        for j, score in enumerate(scores.detach().cpu().numpy()):
            y, x0 = batch_positions[start+j]
            raw_heatmap[row_map[y], col_map[x0]] = max(
                base_score - float(score), 0.0
            )

    max_drop = raw_heatmap.max()
    normalized_heatmap = (
        raw_heatmap / max_drop
        if max_drop > 1e-12 else np.zeros_like(raw_heatmap)
    )

    return (
        normalized_heatmap, raw_heatmap,
        pred_class, base_prob[0].detach().cpu().numpy()
    )

def image_for_display(image_tensor):
    image = image_tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = np.clip(image * 0.5 + 0.5, 0, 1)
    return np.ascontiguousarray((image * 255).astype(np.uint8))

def create_block_heatmap(heatmap, image_shape):
    h, w = image_shape[:2]
    display = cv2.resize(
        heatmap, (w, h), interpolation=cv2.INTER_NEAREST
    )
    return np.clip(display, 0, 1)

def create_occlusion_overlay(image_uint8, heatmap_display):
    heat_u8 = (heatmap_display * 255).astype(np.uint8)
    heat_rgb = cv2.cvtColor(
        cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET),
        cv2.COLOR_BGR2RGB
    )
    overlay = (
        0.30 * image_uint8.astype(np.float32)
        + 0.70 * heat_rgb.astype(np.float32)
    )
    return np.clip(overlay, 0, 255).astype(np.uint8)

selected_indices = []

for class_idx in range(num_classes):
    indices = [
        i for i in range(len(test_ds))
        if int(test_ds[i][1]) == class_idx
    ]
    if not indices:
        print(f"Warning: no test image for {class_names[class_idx]}")
        selected_indices.append(None)
    else:
        selected_indices.append(int(np.random.choice(indices)))

selected_indices = [i for i in selected_indices if i is not None]
if not selected_indices:
    raise ValueError("No valid test images found.")

n_rows = len(selected_indices)
fig = plt.figure(figsize=(13.5, 3.0 * n_rows))

gs = fig.add_gridspec(
    n_rows, 5,
    width_ratios=[1.0, 1.0, 1.0, 0.045, 1.65],
    hspace=0.24,
    wspace=0.10
)

for row, idx in enumerate(selected_indices):
    image_tensor, true_label = test_ds[idx]
    true_label = int(true_label)
    image_uint8 = image_for_display(image_tensor)

    heatmap, raw_heatmap, pred_class, probabilities = (
        occlusion_sensitivity(
            model, image_tensor,
            patch_size=32, stride=16, batch_size=32
        )
    )

    heatmap_display = create_block_heatmap(
        heatmap, image_uint8.shape
    )
    overlay = create_occlusion_overlay(
        image_uint8, heatmap_display
    )

    ax1 = fig.add_subplot(gs[row, 0])
    ax2 = fig.add_subplot(gs[row, 1])
    ax3 = fig.add_subplot(gs[row, 2])
    cax = fig.add_subplot(gs[row, 3])
    ax4 = fig.add_subplot(gs[row, 4])

    ax1.imshow(image_uint8, interpolation="nearest")
    ax2.imshow(overlay, interpolation="nearest")
    im = ax3.imshow(
        heatmap_display, cmap="jet",
        vmin=0, vmax=1, interpolation="nearest",
        aspect="equal"
    )

    ax1.set_title(
        f"Original - {class_names[true_label]}\n"
        f"Pred: {class_names[pred_class]}",
        fontsize=8.5, pad=4
    )
    ax2.set_title("Occlusion Sensitivity", fontsize=9, pad=4)
    ax3.set_title("Heatmap (Normalized)", fontsize=9, pad=4)

    for ax in (ax1, ax2, ax3):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal", adjustable="box")

    cb = fig.colorbar(im, cax=cax)
    cb.set_ticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    cb.ax.tick_params(labelsize=7, length=2, pad=1)

    ax4.axis("off")
    ax4.set_xlim(0, 1)
    ax4.set_ylim(0, 1)

    ax4.text(
        0.06, 0.98, "Prediction Scores (all classes)",
        fontsize=10.5, color="black",
        transform=ax4.transAxes, va="top", ha="left"
    )

    order = np.argsort(probabilities)[::-1]
    y = 0.80

    for class_idx in order:
        color = "green" if class_idx == pred_class else "black"
        ax4.text(
            0.06, y,
            f"{class_names[class_idx]} : {probabilities[class_idx]:.4f}",
            fontsize=9, color=color,
            transform=ax4.transAxes, va="top", ha="left"
        )
        y -= 0.105

fig.subplots_adjust(
    top=0.97, bottom=0.025,
    left=0.015, right=0.99
)
plt.show()

print("\n" + "=" * 65)
print("DENSENET201 + SWIN OCCLUSION SENSITIVITY COMPLETE")
print("=" * 65)

for row, idx in enumerate(selected_indices):
    print(f"{class_names[row]} -> Test index: {idx}")


In [ ]:
import numpy as np
import pandas as pd
import torch
import cv2
from scipy import stats
from skimage.metrics import structural_similarity as ssim
from tqdm.auto import tqdm

model.eval()

PATCH_SIZE = 32
STRIDE = 16
BATCH_SIZE = 32
image_results = []

def get_occlusion_metrics(model, image_tensor):
    x = image_tensor.unsqueeze(0).float().to(device)

    with torch.no_grad():
        output = model(x)
        if isinstance(output, tuple):
            output = output[0]
        probs = torch.softmax(output, dim=1)[0]
        pred_class = int(probs.argmax().item())
        original_conf = float(probs[pred_class].item())

    _, _, h, w = x.shape
    patch = min(PATCH_SIZE, h, w)

    ys = list(range(0, h - patch + 1, STRIDE))
    xs = list(range(0, w - patch + 1, STRIDE))
    if not ys or ys[-1] != h - patch:
        ys.append(h - patch)
    if not xs or xs[-1] != w - patch:
        xs.append(w - patch)

    original = x[0]
    drops = []
    positions = []
    occluded_images = []

    for y in sorted(set(ys)):
        for x0 in sorted(set(xs)):
            occluded = original.clone()
            region = occluded[:, y:y+patch, x0:x0+patch]
            region_mean = region.mean(dim=(1, 2), keepdim=True)
            occluded[:, y:y+patch, x0:x0+patch] = region_mean
            occluded_images.append(occluded)
            positions.append((y, x0))

    for start in range(0, len(occluded_images), BATCH_SIZE):
        batch = torch.stack(
            occluded_images[start:start+BATCH_SIZE]
        ).to(device)

        with torch.no_grad():
            output = model(batch)
            if isinstance(output, tuple):
                output = output[0]
            scores = torch.softmax(output, dim=1)[:, pred_class]

        drops.extend(
            (original_conf - scores).detach().cpu().numpy().tolist()
        )

    sensitivity_score = float(np.mean(np.abs(drops)))

    heatmap = np.zeros((h, w), dtype=np.float32)
    count = np.zeros((h, w), dtype=np.float32)

    for (y, x0), drop in zip(positions, drops):
        heatmap[y:y+patch, x0:x0+patch] += abs(drop)
        count[y:y+patch, x0:x0+patch] += 1

    heatmap = np.divide(
        heatmap, count,
        out=np.zeros_like(heatmap),
        where=count > 0
    )

    max_value = heatmap.max()
    if max_value > 1e-12:
        heatmap = heatmap / max_value

    reference = np.zeros_like(heatmap)
    if np.max(heatmap) > 0:
        reference[:] = np.mean(heatmap)

    if np.std(heatmap) < 1e-12:
        heatmap_ssim = 1.0
    else:
        heatmap_ssim = float(ssim(
            heatmap, reference, data_range=1.0
        ))

    fully_occluded = original.clone()
    for y, x0 in positions:
        fully_occluded[:, y:y+patch, x0:x0+patch] = (
            fully_occluded[:, y:y+patch, x0:x0+patch]
            .mean(dim=(1, 2), keepdim=True)
        )

    with torch.no_grad():
        output = model(fully_occluded.unsqueeze(0))
        if isinstance(output, tuple):
            output = output[0]
        final_conf = float(
            torch.softmax(output, dim=1)[0, pred_class].item()
        )

    confidence_change = (
        (original_conf - final_conf) / max(original_conf, 1e-12)
    ) * 100

    return sensitivity_score, heatmap_ssim, confidence_change, pred_class


try:
    for idx in tqdm(range(len(test_ds)), desc="Occlusion Statistics"):
        image_tensor, true_label = test_ds[idx]

        score, similarity, conf_change, pred_class = get_occlusion_metrics(
            model, image_tensor
        )

        image_results.append({
            "Image_Index": idx,
            "True_Class": class_names[int(true_label)],
            "Predicted_Class": class_names[pred_class],
            "Correct": int(pred_class == int(true_label)),
            "Occlusion_Sensitivity_Score": score,
            "Heatmap_Similarity_SSIM": similarity,
            "Confidence_Change (%)": conf_change
        })

except Exception:
    raise

results_df = pd.DataFrame(image_results)
results_df.to_csv("Occlusion_Image_Level_Results.csv", index=False)

metrics = [
    ("Occlusion Sensitivity Score", "Occlusion_Sensitivity_Score"),
    ("Heatmap Similarity (SSIM)", "Heatmap_Similarity_SSIM"),
    ("Confidence Change (%)", "Confidence_Change (%)")
]

summary = []
for metric_name, column in metrics:
    values = results_df[column].dropna().to_numpy()
    n = len(values)
    mean = np.mean(values)
    sd = np.std(values, ddof=1) if n > 1 else np.nan

    if n > 1:
        margin = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
        lower, upper = mean - margin, mean + margin
    else:
        lower, upper = np.nan, np.nan

    summary.append({
        "Metric": metric_name,
        "N": n,
        "Mean": mean,
        "SD": sd,
        "95% CI Lower": lower,
        "95% CI Upper": upper
    })

table4_df = pd.DataFrame(summary)

print(table4_df.round(4).to_string(index=False))

table4_df.to_csv("Occlusion_Overall_Statistics.csv", index=False)
print("\nSaved: Occlusion_Image_Level_Results.csv")
print("Saved: Occlusion_Overall_Statistics.csv")


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm

model.eval()
results = []

for idx in tqdm(range(len(test_ds)), desc="Class-wise Occlusion"):
    image_tensor, true_label = test_ds[idx]
    true_label = int(true_label)

    sensitivity, heatmap_ssim, confidence_change, pred_class = (
        get_occlusion_metrics(model, image_tensor)
    )

    results.append({
        "Image_Index": idx,
        "True_Class": class_names[true_label],
        "Predicted_Class": class_names[pred_class],
        "Occlusion_Sensitivity_Score": sensitivity
    })

image_results_df = pd.DataFrame(results)

grouped = image_results_df.groupby("True_Class")[
    "Occlusion_Sensitivity_Score"
]

summary = grouped.agg(
    N="count",
    Mean_Sensitivity="mean",
    SD="std"
).reindex(class_names)

ci_lower = []
ci_upper = []

for class_name in class_names:
    values = image_results_df.loc[
        image_results_df["True_Class"] == class_name,
        "Occlusion_Sensitivity_Score"
    ].dropna().to_numpy()

    n = len(values)

    if n > 1:
        mean = np.mean(values)
        sd = np.std(values, ddof=1)
        margin = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n)
        ci_lower.append(mean - margin)
        ci_upper.append(mean + margin)
    else:
        ci_lower.append(np.nan)
        ci_upper.append(np.nan)

summary["95% CI Lower"] = ci_lower
summary["95% CI Upper"] = ci_upper
summary = summary.reset_index(names="Class")

print(summary.round(5).to_string(index=False))

summary.to_csv("Occlusion_Classwise_Statistics.csv", index=False)
image_results_df.to_csv("Occlusion_Classwise_Image_Level.csv", index=False)

print("\nSaved: Occlusion_Classwise_Statistics.csv")
print("Saved: Occlusion_Classwise_Image_Level.csv")


In [ ]:
import numpy as np
import pandas as pd
import torch
from scipy import stats
from skimage.metrics import structural_similarity as ssim
from tqdm.auto import tqdm

model.eval()

NOISE_LEVELS = {
    "Level 0": 0.00,
    "Level 1": 0.02,
    "Level 2": 0.05,
    "Level 3": 0.10,
    "Level 4": 0.15
}

def to_display_rgb(tensor):
    img = tensor.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(img * 0.5 + 0.5, 0, 1).astype(np.float32)

def to_model_tensor(img):
    tensor = torch.from_numpy(img).permute(2, 0, 1).float()
    tensor = (tensor - 0.5) / 0.5
    return tensor

def perturbation_evaluation(model, image_tensor, noise_std):
    original = to_display_rgb(image_tensor)
    x = to_model_tensor(original).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(x)
        if isinstance(output, tuple):
            output = output[0]
        prob = torch.softmax(output, dim=1)[0]
        pred_class = int(prob.argmax().item())
        original_conf = float(prob[pred_class].item())

    if noise_std == 0:
        perturbed = original.copy()
    else:
        rng = np.random.default_rng()
        noise = rng.normal(0, noise_std, original.shape).astype(np.float32)
        perturbed = np.clip(original + noise, 0, 1)

    perturbed_tensor = to_model_tensor(perturbed).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(perturbed_tensor)
        if isinstance(output, tuple):
            output = output[0]
        perturbed_prob = torch.softmax(output, dim=1)[0]
        perturbed_conf = float(perturbed_prob[pred_class].item())

    sensitivity = abs(original_conf - perturbed_conf)

    similarity = ssim(
        original, perturbed,
        data_range=1.0, channel_axis=2
    )

    return sensitivity, similarity, original_conf, perturbed_conf, pred_class


records = []

for level_name, noise_std in NOISE_LEVELS.items():
    for idx in tqdm(range(len(test_ds)), desc=level_name):
        image_tensor, true_label = test_ds[idx]

        sensitivity, similarity, base_conf, pert_conf, pred_class = (
            perturbation_evaluation(model, image_tensor, noise_std)
        )

        records.append({
            "Image_Index": idx,
            "True_Class": class_names[int(true_label)],
            "Predicted_Class": class_names[pred_class],
            "Perturbation_Level": level_name,
            "Noise_Std": noise_std,
            "Sensitivity_Score": sensitivity,
            "SSIM": similarity,
            "Original_Confidence": base_conf,
            "Perturbed_Confidence": pert_conf
        })

results_df = pd.DataFrame(records)
results_df.to_csv("Perturbation_Level_Image_Results.csv", index=False)

summary = []

for level_name in NOISE_LEVELS:
    subset = results_df[
        results_df["Perturbation_Level"] == level_name
    ]

    values = subset["Sensitivity_Score"].dropna().to_numpy()
    n = len(values)
    mean_sens = np.mean(values)
    sd_sens = np.std(values, ddof=1) if n > 1 else np.nan
    mean_ssim = subset["SSIM"].mean()

    summary.append({
        "Perturbation Level": level_name,
        "N": n,
        "Mean Sensitivity": mean_sens,
        "SD": sd_sens,
        "Mean SSIM": mean_ssim
    })

table6_df = pd.DataFrame(summary)

print(table6_df.round(5).to_string(index=False))

table6_df.to_csv("Perturbation_Level_Summary.csv", index=False)

print("\nSaved: Perturbation_Level_Image_Results.csv")
print("Saved: Perturbation_Level_Summary.csv")
